# PM2.5 Autoregressive Model - Data Preparation & Analysis

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h3
import folium
from folium.plugins import MeasureControl
import os
import warnings
warnings.filterwarnings('ignore')

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Data Loading & Coverage Analysis

In [17]:
df = pd.read_csv('../../data/pm25_enriched_hourly_2023_20250826_004834.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unique hexagons: {df['hex7_id'].nunique()}")

Dataset shape: (2049356, 17)
Date range: 2023-07-14 16:00:00+00:00 to 2023-12-31 23:00:00+00:00
Unique hexagons: 629


In [18]:
coverage_stats = df.groupby('hex7_id').agg({
    'pm25_ugm3_mean': ['count', lambda x: x.notna().sum(), lambda x: x.notna().mean()]
}).round(3)

coverage_stats.columns = ['total_hours', 'valid_pm25', 'coverage_ratio']

print(f"\nFull Dataset Coverage Summary:")
print(coverage_stats.describe())


Full Dataset Coverage Summary:
       total_hours  valid_pm25  coverage_ratio
count      629.000     629.000         629.000
mean      2585.038    2585.038           0.789
std       1257.449    1257.449           0.364
min          0.000       0.000           0.000
25%       2279.000    2279.000           0.869
50%       3272.000    3272.000           0.968
75%       3367.000    3367.000           0.987
max       3460.000    3460.000           1.000


## 2. H3 Resolution Analysis

In [ ]:
def aggregate_to_resolution(df, target_resolution):
    df_agg = df.copy()
    df_agg['parent_hex'] = df_agg['hex7_id'].apply(lambda x: h3.cell_to_parent(x, target_resolution))
    
    result = df_agg.groupby(['timestamp', 'parent_hex']).agg({
        'pm25_ugm3_mean': ['mean', 'std', 'count', lambda x: x.notna().mean()]
    }).reset_index()
    
    result.columns = ['timestamp', 'hex_id', 'pm25_mean', 'pm25_std', 'sensor_count', 'coverage']
    result['resolution'] = target_resolution
    
    return result

resolutions = {}
for res in [7, 6, 5]:
    resolutions[res] = aggregate_to_resolution(df, res)
    coverage = resolutions[res].groupby('hex_id')['coverage'].mean().mean()
    hexagons = resolutions[res]['hex_id'].nunique()
    print(f"Resolution {res}: {hexagons} hexagons, {coverage:.1%} avg coverage")

Resolution 7: 629 hexagons, 78.9% avg coverage
Resolution 6: 544 hexagons, 79.3% avg coverage


## 3. Tokyo Focus Analysis

In [ ]:
TOKYO_BOUNDS = {
    'lat_min': 35.5, 'lat_max': 35.9,
    'lon_min': 139.5, 'lon_max': 140.0
}

res7_data = resolutions[7]

res7_with_coords = res7_data.copy()
res7_with_coords['lat'] = res7_with_coords['hex_id'].apply(lambda x: h3.cell_to_latlng(x)[0])
res7_with_coords['lon'] = res7_with_coords['hex_id'].apply(lambda x: h3.cell_to_latlng(x)[1])

tokyo_data = res7_with_coords[
    (res7_with_coords['lat'] >= TOKYO_BOUNDS['lat_min']) &
    (res7_with_coords['lat'] <= TOKYO_BOUNDS['lat_max']) &
    (res7_with_coords['lon'] >= TOKYO_BOUNDS['lon_min']) &
    (res7_with_coords['lon'] <= TOKYO_BOUNDS['lon_max'])
]

tokyo_wide = tokyo_data.pivot_table(
    index='timestamp',
    columns='hex_id',
    values='pm25_mean'
)

print(f"Tokyo area: {tokyo_wide.shape[1]} hexagons, {tokyo_wide.shape[0]} hours")
print(f"Data completeness: {tokyo_wide.notna().mean().mean():.1%}")
print(f"Date range: {tokyo_wide.index.min()} to {tokyo_wide.index.max()}")

## 4. Time Series Feature Engineering

In [ ]:
def create_features(series, hex_id):
    df = pd.DataFrame({'pm25': series})
    df['hexagon'] = hex_id
    df['timestamp'] = df.index
    
    for lag in [1, 2, 3, 4, 5, 6, 12, 24]:
        df[f'lag_{lag}h'] = df['pm25'].shift(lag)
    
    df['rolling_mean_6h'] = df['pm25'].rolling(6, min_periods=1).mean()
    df['rolling_std_6h'] = df['pm25'].rolling(6, min_periods=1).std()
    df['rolling_mean_24h'] = df['pm25'].rolling(24, min_periods=1).mean()
    
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    return df

high_coverage_hexagons = tokyo_wide.columns[tokyo_wide.notna().mean() > 0.8]
print(f"Processing {len(high_coverage_hexagons)} high coverage hexagons")

featured_data = []
for hex_id in high_coverage_hexagons[:10]:
    hex_features = create_features(tokyo_wide[hex_id], hex_id)
    featured_data.append(hex_features)

featured_df = pd.concat(featured_data, ignore_index=True)

## 5. Autocorrelation Analysis

In [ ]:
sample_hex = high_coverage_hexagons[0]
sample_series = tokyo_wide[sample_hex].dropna()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(sample_series.values[:168])
axes[0, 0].set_title('PM2.5 Time Series (1 week)', fontsize=12)
axes[0, 0].set_xlabel('Hours')
axes[0, 0].set_ylabel('PM2.5 (μg/m³)')
axes[0, 0].grid(True, alpha=0.3)

plot_acf(sample_series.values, lags=48, ax=axes[0, 1])
axes[0, 1].set_title('Autocorrelation Function (ACF)', fontsize=12)
axes[0, 1].set_xlabel('Lag (hours)')

plot_pacf(sample_series.values, lags=48, ax=axes[1, 0])
axes[1, 0].set_title('Partial Autocorrelation Function (PACF)', fontsize=12)
axes[1, 0].set_xlabel('Lag (hours)')

autocorr_values = [sample_series.autocorr(lag=i) for i in range(1, 49)]
axes[1, 1].scatter(range(1, 49), autocorr_values, alpha=0.6)
axes[1, 1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1, 1].axhline(y=1.96/np.sqrt(len(sample_series)), color='r', linestyle='--', alpha=0.5)
axes[1, 1].axhline(y=-1.96/np.sqrt(len(sample_series)), color='r', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Autocorrelation by Lag', fontsize=12)
axes[1, 1].set_xlabel('Lag (hours)')
axes[1, 1].set_ylabel('Autocorrelation')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Time Series Analysis for Hexagon {sample_hex[:8]}...', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nKey observations:")
print(f"ACF at 24h lag: {sample_series.autocorr(lag=24):.3f}")
print(f"ACF at 12h lag: {sample_series.autocorr(lag=12):.3f}")
print(f"ACF at 6h lag: {sample_series.autocorr(lag=6):.3f}")
print(f"ACF at 1h lag: {sample_series.autocorr(lag=1):.3f}")

## 6. Multi-Horizon Target Creation

In [ ]:
ml_datasets = []

for hex_id in high_coverage_hexagons[:10]:
    hex_data = featured_df[featured_df['hexagon'] == hex_id].copy()
    
    for h in [1, 2, 3, 4, 5, 6, 12, 24]:
        hex_data[f'target_h{h}'] = hex_data['pm25'].shift(-h)
    
    ml_datasets.append(hex_data)

ml_dataset = pd.concat(ml_datasets, ignore_index=True)
ml_dataset = ml_dataset.dropna()

print(f"ML dataset: {ml_dataset.shape[0]} samples, {ml_dataset.shape[1]} features")

## 7. Train/Test Split

In [ ]:
ml_dataset = ml_dataset.sort_values('timestamp').reset_index(drop=True)

unique_timestamps = ml_dataset['timestamp'].unique()
n_timestamps = len(unique_timestamps)

train_end_idx = int(0.8 * n_timestamps)
train_end_time = unique_timestamps[train_end_idx]

train = ml_dataset[ml_dataset['timestamp'] < train_end_time].copy()
test = ml_dataset[ml_dataset['timestamp'] >= train_end_time].copy()

print(f"Train: {len(train):,} samples")
print(f"Test: {len(test):,} samples")
print(f"\nTrain period: {train['timestamp'].min()} to {train['timestamp'].max()}")
print(f"Test period: {test['timestamp'].min()} to {test['timestamp'].max()}")

## 8. Model Comparison

In [ ]:
feature_cols = [c for c in train.columns if c not in ['target_h1', 'target_h2', 'target_h3', 'target_h4', 
                                                        'target_h5', 'target_h6', 'target_h12', 'target_h24',
                                                        'hexagon', 'timestamp', 'pm25']]

X_train = train[feature_cols]
X_test = test[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

horizons = [1, 2, 3, 4, 5, 6, 12, 24]
models = {
    'Persistence': None,
    'LinearReg': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=50, max_depth=5, random_state=42)
}

results = []

for h in horizons:
    target_col = f'target_h{h}'
    y_train_h = train[target_col]
    y_test_h = test[target_col]
    
    for model_name, model in models.items():
        if model_name == 'Persistence':
            lag_col = f'lag_{h}h' if f'lag_{h}h' in X_test.columns else 'lag_1h'
            predictions = X_test[lag_col].values
        else:
            if model_name in ['Ridge']:
                model.fit(X_train_scaled, y_train_h)
                predictions = model.predict(X_test_scaled)
            else:
                model.fit(X_train, y_train_h)
                predictions = model.predict(X_test)
        
        mae = mean_absolute_error(y_test_h, predictions)
        rmse = np.sqrt(mean_squared_error(y_test_h, predictions))
        r2 = r2_score(y_test_h, predictions)
        
        results.append({
            'Horizon': f'{h}h',
            'Model': model_name,
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2
        })

results_df = pd.DataFrame(results)

pivot_mae = results_df.pivot_table(index='Model', columns='Horizon', values='MAE')
pivot_mae = pivot_mae[[f'{h}h' for h in horizons if f'{h}h' in pivot_mae.columns]]

pivot_r2 = results_df.pivot_table(index='Model', columns='Horizon', values='R2')
pivot_r2 = pivot_r2[[f'{h}h' for h in horizons if f'{h}h' in pivot_r2.columns]]

print("\nMAE by Model and Horizon (μg/m³):")
print(pivot_mae.round(2))

print("\nR² by Model and Horizon:")
print(pivot_r2.round(3))

## 9. Map Modeling & Visualization

In [ ]:
print("\nCreating combined comparison map for all Japan 2023 data with sensor locations...")

all_hexagons = df['hex7_id'].unique()
print(f"Total unique hexagons: {len(all_hexagons)}")

df_coords = df.copy()
df_coords['lat'] = df_coords['hex7_id'].apply(lambda x: h3.cell_to_latlng(x)[0])
df_coords['lon'] = df_coords['hex7_id'].apply(lambda x: h3.cell_to_latlng(x)[1])

center_lat = df_coords['lat'].mean()
center_lon = df_coords['lon'].mean()

print(f"Map center: {center_lat:.4f}°N, {center_lon:.4f}°E")

comparison_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=5,
    tiles='OpenStreetMap',
    control_scale=True
)

comparison_map.add_child(MeasureControl(primary_length_unit='kilometers'))

sensor_locations = df[['hex7_id']].drop_duplicates()
sensor_locations['lat'] = sensor_locations['hex7_id'].apply(lambda x: h3.cell_to_latlng(x)[0])
sensor_locations['lon'] = sensor_locations['hex7_id'].apply(lambda x: h3.cell_to_latlng(x)[1])

sensor_layer = folium.FeatureGroup(name='📍 Original Sensor Locations', show=True)
for _, row in sensor_locations.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='darkblue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.8,
        weight=1,
        popup=f"Sensor Location<br>Lat: {row['lat']:.4f}<br>Lon: {row['lon']:.4f}<br>H3: {row['hex7_id']}",
        tooltip=f"Sensor: {row['lat']:.2f}°N, {row['lon']:.2f}°E"
    ).add_to(sensor_layer)
sensor_layer.add_to(comparison_map)

In [ ]:
res7_hexagons = {}
for sensor in all_hexagons:
    parent = h3.cell_to_parent(sensor, 7)
    res7_hexagons[parent] = res7_hexagons.get(parent, 0) + 1

res7_layer = folium.FeatureGroup(name='Resolution 7 (5.16 km²)', show=True)
for hex_id, count in res7_hexagons.items():
    try:
        boundary = h3.cell_to_boundary(hex_id)
        color = '#00FF00' if count == 1 else '#FFFF00'
        fill_opacity = 0.3 if count == 1 else 0.5
        folium.Polygon(
            locations=boundary,
            color=color,
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=fill_opacity,
            tooltip=f"Res 7: {count} sensors, 5.16 km²"
        ).add_to(res7_layer)
    except:
        continue
res7_layer.add_to(comparison_map)

res6_hexagons = {}
for sensor in all_hexagons:
    parent = h3.cell_to_parent(sensor, 6)
    res6_hexagons[parent] = res6_hexagons.get(parent, 0) + 1

res6_layer = folium.FeatureGroup(name='Resolution 6 (36.13 km²)', show=False)
for hex_id, count in res6_hexagons.items():
    try:
        boundary = h3.cell_to_boundary(hex_id)
        color = '#00FF00' if count == 1 else '#FFFF00'
        fill_opacity = 0.3 if count == 1 else 0.5
        folium.Polygon(
            locations=boundary,
            color=color,
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=fill_opacity,
            tooltip=f"Res 6: {count} sensors, 36.13 km²"
        ).add_to(res6_layer)
    except:
        continue
res6_layer.add_to(comparison_map)

In [ ]:
res5_hexagons = {}
for sensor in all_hexagons:
    parent = h3.cell_to_parent(sensor, 5)
    res5_hexagons[parent] = res5_hexagons.get(parent, 0) + 1

res5_layer = folium.FeatureGroup(name='Resolution 5 (252.9 km²)', show=False)
for hex_id, count in res5_hexagons.items():
    try:
        boundary = h3.cell_to_boundary(hex_id)
        color = '#00FF00' if count == 1 else '#FFFF00'
        fill_opacity = 0.3 if count == 1 else 0.5
        folium.Polygon(
            locations=boundary,
            color=color,
            weight=2,
            fill=True,
            fillColor=color,
            fillOpacity=fill_opacity,
            tooltip=f"Res 5: {count} sensors, 252.9 km²"
        ).add_to(res5_layer)
    except:
        continue
res5_layer.add_to(comparison_map)

res8_layer = folium.FeatureGroup(name='Resolution 8 (0.74 km²)', show=False)
sample_sensors = all_hexagons[:500] if len(all_hexagons) > 500 else all_hexagons
for hex_id in sample_sensors:
    try:
        boundary = h3.cell_to_boundary(hex_id)
        folium.Polygon(
            locations=boundary,
            color='#00FF00',
            weight=1,
            fill=True,
            fillColor='#00FF00',
            fillOpacity=0.3,
            tooltip=f"Res 8: 1 sensor, 0.74 km²"
        ).add_to(res8_layer)
    except:
        continue
res8_layer.add_to(comparison_map)

folium.LayerControl(collapsed=False).add_to(comparison_map)

In [ ]:
title_html = f'''
<script>
function toggleInfo() {{
    var content = document.getElementById('info-content');
    var icon = document.getElementById('toggle-icon');
    var box = document.getElementById('info-box');
    
    if (content.style.display === 'none') {{
        content.style.display = 'block';
        icon.innerHTML = '−';
        box.style.width = '450px';
    }} else {{
        content.style.display = 'none';
        icon.innerHTML = '+';
        box.style.width = 'auto';
    }}
}}
</script>

<div id="info-box" style="position: fixed; 
            top: 70px; 
            left: 10px; 
            width: 450px; 
            background: rgba(255, 255, 255, 0.95); 
            z-index: 1000;
            padding: 10px;
            border: 2px solid rgba(0,0,0,0.3);
            border-radius: 5px;
            font-size: 12px;">
    <div style="display: flex; justify-content: space-between; align-items: center;">
        <h4 style="margin: 0;">Japan PM2.5 Coverage Map - 2023</h4>
        <button onclick="toggleInfo()" style="background: none; border: none; font-size: 20px; cursor: pointer;" id="toggle-icon">−</button>
    </div>
</div>
'''

comparison_map.get_root().html.add_child(folium.Element(title_html))

os.makedirs('maps', exist_ok=True)
comparison_map.save('maps/japan_2023_coverage_comparison_with_sensors.html')
print(f"Saved as maps/japan_2023_coverage_comparison_with_sensors.html")
print(f"Total sensor locations: {len(sensor_locations)}")
print(f"Geographic extent: {df_coords['lat'].min():.1f}°N-{df_coords['lat'].max():.1f}°N, {df_coords['lon'].min():.1f}°E-{df_coords['lon'].max():.1f}°E")

comparison_map